# Notebook 1 — Exploratory Data Analysis
LendingClub Accepted Loans (2007–2018)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Data

In [ ]:
CSV_PATH = '../accepted_2007_to_2018Q4.csv'
SAMPLE_SIZE = 200_000

df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f'Full dataset shape: {df_raw.shape}')
print(f'Columns: {df_raw.shape[1]}')

In [ ]:
# Keep only terminal loan statuses
df = df_raw[df_raw['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
print(f'After filtering terminal statuses: {len(df):,} rows')

# Sample for analysis
df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f'Working sample: {len(df):,} rows')

## 2. Target Variable

In [ ]:
df['target'] = (df['loan_status'] == 'Charged Off').astype(int)

counts = df['loan_status'].value_counts()
print(counts)
print(f'\nDefault rate: {df["target"].mean():.2%}')

In [ ]:
counts = df['loan_status'].value_counts()
print(counts)
print(f'\nDefault rate: {df["target"].mean():.2%}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax.set_title('Loan Status Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Missing Values

In [ ]:
KEEP_COLS = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies',
    'purpose', 'home_ownership', 'verification_status', 'term',
    'grade', 'sub_grade', 'inq_last_6mths', 'emp_length', 'earliest_cr_line',
]

df_feat = df[KEEP_COLS + ['target']].copy()

missing = df_feat.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_feat) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})[missing > 0]

## 4. Numeric Feature Distributions

In [ ]:
# Clean numeric cols first
df_feat['int_rate'] = df_feat['int_rate'].astype(str).str.replace('%','').astype(float)
df_feat['revol_util'] = pd.to_numeric(df_feat['revol_util'].astype(str).str.replace('%',''), errors='coerce')
df_feat['term'] = df_feat['term'].astype(str).str.strip().str.replace(' months','').astype(int)

df_feat.describe().T[['mean','std','min','50%','max']]

In [ ]:
num_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'revol_util']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    paid  = df_feat.loc[df_feat['target']==0, col].dropna()
    choff = df_feat.loc[df_feat['target']==1, col].dropna()
    axes[i].hist(paid,  bins=40, alpha=0.5, label='Fully Paid',  color='#2ecc71')
    axes[i].hist(choff, bins=40, alpha=0.5, label='Charged Off', color='#e74c3c')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Loan Status', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Grade vs Default Rate

In [ ]:
grade_default = df_feat.groupby('grade')['target'].mean().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(grade_default.index, grade_default.values,
              color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(grade_default))),
              edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, grade_default.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Loan Grade', fontsize=12)
ax.set_ylabel('Default Rate', fontsize=12)
ax.set_title('Default Rate by Loan Grade', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Purpose vs Default Rate

In [ ]:
purpose_default = (
    df_feat.groupby('purpose')['target']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'default_rate', 'count': 'loan_count'})
    .sort_values('default_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(purpose_default)))
bars = ax.barh(purpose_default['purpose'], purpose_default['default_rate'],
               color=colors, edgecolor='white')
for bar, val in zip(bars, purpose_default['default_rate']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Default Rate', fontsize=12)
ax.set_title('Default Rate by Loan Purpose', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

In [ ]:
corr_cols = ['loan_amnt','int_rate','annual_inc','dti','fico_range_low',
             'revol_util','delinq_2yrs','inq_last_6mths','open_acc','target']

corr = df_feat[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.zeros_like(corr, dtype=bool)
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, ax=ax, mask=mask,
    linewidths=0.5, annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. FICO Score vs Default Rate

In [ ]:
df_feat['fico_bin'] = pd.cut(df_feat['fico_range_low'], bins=range(580, 870, 20))
fico_default = df_feat.groupby('fico_bin', observed=True)['target'].mean().reset_index()
fico_default['fico_mid'] = fico_default['fico_bin'].apply(lambda x: x.mid)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fico_default['fico_mid'], fico_default['target'],
        color='#e74c3c', lw=2.5, marker='o', markersize=5)
ax.fill_between(fico_default['fico_mid'], fico_default['target'], alpha=0.15, color='#e74c3c')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('FICO Score', fontsize=12)
ax.set_ylabel('Default Rate', fontsize=12)
ax.set_title('Default Rate by FICO Score Band', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. DTI vs Loan Amount Scatter

In [ ]:
sample = df_feat.sample(5000, random_state=42)
colors = sample['target'].map({0: '#2ecc71', 1: '#e74c3c'})

fig, ax = plt.subplots(figsize=(8, 5))
for label, color, marker in [(0, '#2ecc71', 'o'), (1, '#e74c3c', 'x')]:
    mask = sample['target'] == label
    ax.scatter(
        sample.loc[mask, 'dti'], sample.loc[mask, 'loan_amnt'],
        c=color, alpha=0.3, s=15, label=['Fully Paid', 'Charged Off'][label],
        marker=marker
    )
ax.set_xlabel('Debt-to-Income Ratio', fontsize=12)
ax.set_ylabel('Loan Amount ($)', fontsize=12)
ax.set_title('DTI vs Loan Amount (5k sample)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 10. Summary Insights

In [ ]:
print('=== Key EDA Findings ===')
print(f'  Default rate       : {df_feat["target"].mean():.2%}')
print(f'  Median loan amount : ${df_feat["loan_amnt"].median():,.0f}')
print(f'  Median int rate    : {df_feat["int_rate"].median():.1f}%')
print(f'  Median FICO        : {df_feat["fico_range_low"].median():.0f}')
print(f'  Grade A default    : {df_feat[df_feat["grade"]=="A"]["target"].mean():.2%}')
print(f'  Grade G default    : {df_feat[df_feat["grade"]=="G"]["target"].mean():.2%}')